In [0]:
from pyspark.sql.functions import col, to_date, trim, lower

bronze_path = "/Volumes/workspace/retail_raw/bronze/"
silver_path = "/Volumes/workspace/retail_raw/silver/"

orders_bronze = spark.read.format("delta").load(bronze_path + "orders")

orders_silver = (
    orders_bronze
    .dropDuplicates(["order_id"])
    .withColumn("order_status", lower(trim(col("order_status"))))
    .withColumn("order_purchase_date", to_date(col("order_purchase_timestamp")))
    .filter(col("order_id").isNotNull())
)

orders_silver.write.format("delta").mode("overwrite").save(silver_path + "orders")

print(f"Silver orders: {orders_silver.count()} rows")
orders_silver.select("order_id", "order_status", "order_purchase_date").show(5)

Silver orders: 99441 rows
+--------------------+------------+-------------------+
|            order_id|order_status|order_purchase_date|
+--------------------+------------+-------------------+
|2807d0e504d6d4894...|   delivered|         2018-02-03|
|ccbabeb0b02433bd0...|   delivered|         2018-02-19|
|e4de6d53ecff736bc...|   delivered|         2017-10-16|
|cadbb3657dac2dbbd...|   delivered|         2018-02-27|
|d3d6788577c9592da...|   delivered|         2017-09-19|
+--------------------+------------+-------------------+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import col

order_items_bronze = spark.read.format("delta").load(bronze_path + "order_items")

order_items_silver = (
    order_items_bronze
    .dropDuplicates(["order_id", "order_item_id"])
    .filter(col("order_id").isNotNull())
    .filter(col("price") > 0)
)

order_items_silver.write.format("delta").mode("overwrite").save(silver_path + "order_items")

print(f"Silver order_items: {order_items_silver.count()} rows")
order_items_silver.select("order_id", "product_id", "price", "freight_value").show(5)

Silver order_items: 112650 rows
+--------------------+--------------------+------+-------------+
|            order_id|          product_id| price|freight_value|
+--------------------+--------------------+------+-------------+
|0010b2e5201cc5f1a...|5a419dbf24a8c9718...|  48.9|         16.6|
|0028de0ca693a1bb2...|059344baebbeaa42f...| 29.99|        15.31|
|0032d07457ae9c806...|08279c494018541f7...| 159.0|        27.19|
|00471463a6106056c...|9df0e8a7eef2a38b7...|179.99|        85.97|
|005d9a5423d47281a...|fb7a100ec8c7b34f6...| 49.99|        18.12|
+--------------------+--------------------+------+-------------+
only showing top 5 rows


In [0]:
customers_bronze = spark.read.format("delta").load(bronze_path + "customers")

customers_silver = (
    customers_bronze
    .dropDuplicates(["customer_id"])
    .withColumn("customer_state", trim(col("customer_state")))
    .filter(col("customer_id").isNotNull())
)

customers_silver.write.format("delta").mode("overwrite").save(silver_path + "customers")

print(f"Silver customers: {customers_silver.count()} rows")
customers_silver.select("customer_id", "customer_city", "customer_state").show(5)

Silver customers: 99441 rows
+--------------------+--------------+--------------+
|         customer_id| customer_city|customer_state|
+--------------------+--------------+--------------+
|237098a64674ae89b...|      curitiba|            PR|
|e3109970a3fe8021d...|     sao paulo|            SP|
|c532a74a3ebf1bacc...|ribeirao preto|            SP|
|19cecb194f54e614b...|belo horizonte|            MG|
|c82a5e4fafdbeb34f...| foz do iguacu|            PR|
+--------------------+--------------+--------------+
only showing top 5 rows


In [0]:
products_bronze = spark.read.format("delta").load(bronze_path + "products")

products_silver = (
    products_bronze
    .dropDuplicates(["product_id"])
    .filter(col("product_id").isNotNull())
)

products_silver.write.format("delta").mode("overwrite").save(silver_path + "products")

print(f"Silver products: {products_silver.count()} rows")
products_silver.select("product_id", "product_category_name").show(5)

Silver products: 32951 rows
+--------------------+---------------------+
|          product_id|product_category_name|
+--------------------+---------------------+
|e1d1d22e9f8122a4e...|   ferramentas_jardim|
|ce5b91848b91118da...|        esporte_lazer|
|1c6fb703c624b381a...|           brinquedos|
|4e04ffb7dd3739ecf...|            papelaria|
|0992c6cba95a13bfa...|   ferramentas_jardim|
+--------------------+---------------------+
only showing top 5 rows


In [0]:
order_payments_bronze = spark.read.format("delta").load(bronze_path + "order_payments")

order_payments_silver = (
    order_payments_bronze
    .dropDuplicates(["order_id", "payment_sequential"])
    .filter(col("order_id").isNotNull())
    .filter(col("payment_value") >= 0)
)

order_payments_silver.write.format("delta").mode("overwrite").save(silver_path + "order_payments")

print(f"Silver order_payments: {order_payments_silver.count()} rows")
order_payments_silver.select("order_id", "payment_type", "payment_installments", "payment_value").show(5)

Silver order_payments: 103886 rows
+--------------------+------------+--------------------+-------------+
|            order_id|payment_type|payment_installments|payment_value|
+--------------------+------------+--------------------+-------------+
|cf30fe76d1505192a...| credit_card|                   2|        47.72|
|85eef2d342b0de363...| credit_card|                   3|        99.43|
|ad4098a257676ea4d...| credit_card|                   2|        68.49|
|fa2ea4b6e84c1c0fc...|  debit_card|                   1|       227.12|
|3ba2d0012b1f34bc6...| credit_card|                   6|        60.07|
+--------------------+------------+--------------------+-------------+
only showing top 5 rows
